# Fase 1: Data Wrangling y Limpieza


## 1. Carga e Inspección Inicial

Primero importamos las librerías necesarias y cargamos el dataset:


In [1]:
import pandas as pd
import numpy as np
import re

In [5]:
df = pd.read_csv("../data/raw/train.csv", low_memory=False)
df.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [6]:
# Eliminamos columnas identitarias que no aportan valor predictivo
cols_to_drop = ["ID", "Customer_ID", "Name", "SSN", "Month"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

## 2. Limpieza de Cadenas en Columnas Numéricas

Varias columnas numéricas contienen caracteres parásito (ejemplo: '28*' en la edad o '19114.12*' en ingresos). Definimos una función con expresiones regulares (regex) para limpiar cualquier texto y convertir el resultado a flotante.


In [7]:
def clean_numeric_string(val):
    """
    Conserva solo dígitos, puntos decimales y el signo menos.
    Convierte el resto en NaN o en float.
    """
    if pd.isna(val):
        return np.nan

    # Extraer caracteres válidos para números
    cleaned = re.sub(r"[^0-9.-]", "", str(val))

    try:
        return float(cleaned) if cleaned != "" else np.nan
    except ValueError:
        return np.nan


# Lista de columnas afectadas por ruido de texto
numeric_noisy_cols = [
    "Age",
    "Annual_Income",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Outstanding_Debt",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

# Aplicamos la función a las columnas objetivo
for col in numeric_noisy_cols:
    df[col] = df[col].apply(clean_numeric_string)

## 3. Corrección de Outliers y Valores Incoherentes

Una vez convertidas a número, identificaremos valores ilógicos (como edades de 300 años o números de préstamos negativos) y los transformaremos en NaN para imputarlos correctamente más adelante.


In [8]:
# Edades coherentes para crédito bancario (ej: entre 18 y 100 años)
df["Age"] = df["Age"].apply(lambda x: x if (18 <= x <= 100) else np.nan)

# Valores que no pueden ser negativos
non_negative_cols = [
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Num_Credit_Card",
    "Outstanding_Debt",
]

for col in non_negative_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if (pd.notna(x) and x >= 0) else np.nan)

## 4. Transformación de "Credit_History_Age" a Meses Totales

La columna Credit_History_Age viene formateada como un texto (ej: "22 Years and 1 Months"). Para que el algoritmo lo entienda, debemos convertirlo a una sola variable numérica: meses totales.


In [9]:
def parse_credit_history(val):
    if pd.isna(val) or not isinstance(val, str):
        return np.nan

    years = re.search(r"(\d+)\s*Years?", val)
    months = re.search(r"(\d+)\s*Months?", val)

    total_months = 0
    if years:
        total_months += int(years.group(1)) * 12
    if months:
        total_months += int(months.group(1))

    return total_months if (years or months) else np.nan


# Creamos la nueva variable numérica y eliminamos la original
df["Credit_History_Age_Months"] = df["Credit_History_Age"].apply(parse_credit_history)
df = df.drop(columns=["Credit_History_Age"])

## 5. Mapeo Ordinal de la Variable Objetivo (Credit_Score)

El target viene como 'Poor', 'Standard' y 'Good'. Lo transformamos respetando su jerarquía explícita:


In [10]:
target_map = {"Poor": 0, "Standard": 1, "Good": 2}

df["Credit_Score"] = df["Credit_Score"].map(target_map)

## ¿Qué logramos con esta Fase 1?

Pasamos de tener un DataFrame contaminado con strings a un DataFrame casi 100% numérico y tipado correctamente.

Los datos atípicos o ruidosos ahora son NaN limpios, listos para ser gestionados mediante imputación dentro de un Pipeline de Scikit-Learn en la siguiente fase (así evitamos la fuga de información/data leakage).


In [11]:
df.head()

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Credit_History_Age_Months
0,23.0,Scientist,19114.12,1824.843333,3,4,3,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,...,_,809.98,26.822620,No,49.574949,80.415295,High_spent_Small_value_payments,312.494089,2,265.0
1,23.0,Scientist,19114.12,NaN,3,4,3,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",-1,...,Good,809.98,31.944960,No,49.574949,118.280222,Low_spent_Large_value_payments,284.629162,2,NaN
2,NaN,Scientist,19114.12,NaN,3,4,3,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,...,Good,809.98,28.609352,No,49.574949,81.699521,Low_spent_Medium_value_payments,331.209863,2,267.0
3,23.0,Scientist,19114.12,NaN,3,4,3,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5,...,Good,809.98,31.377862,No,49.574949,199.458074,Low_spent_Small_value_payments,223.451310,2,268.0
4,23.0,Scientist,19114.12,1824.843333,3,4,3,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6,...,Good,809.98,24.797347,No,49.574949,41.420153,High_spent_Medium_value_payments,341.489231,2,269.0


In [12]:
df.to_csv("../data/Fase1_Output.csv", index=False)